# Part B - Custom Retrieval System
**Student:** Dabone Abdoul Latif  
**Index:** 10012200015  

This notebook handles the implementation of the retrieval system, including embeddings, vector storage, and hybrid search.

In [ ]:
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# 1. Load Chunks
with open('chunks/csv_chunks.json', 'r') as f:
    csv_chunks = json.load(f)
with open('chunks/pdf_chunks.json', 'r') as f:
    pdf_chunks = json.load(f)
all_chunks = csv_chunks + pdf_chunks
texts = [c['text'] for c in all_chunks]

# 2. Generate Embeddings (Task 1)
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(texts, show_progress_bar=True)

# 3. FAISS Vector Store (Task 2)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))
os.makedirs('indexes', exist_ok=True)
faiss.write_index(index, 'indexes/rag_index.faiss')

# 4. BM25 Setup
tokenized_corpus = [text.lower().split() for text in texts]
bm25 = BM25Okapi(tokenized_corpus)

print(f"Retrieved {len(all_chunks)} chunks.")
print(f"FAISS index saved with {index.ntotal} vectors.")

### Hybrid Search Implementation (Task 3 & 4)

In [ ]:
def hybrid_search(query, k=3, alpha=0.5):
    # Vector Search
    query_vec = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, k * 2)
    
    # BM25 Search
    bm25_scores = bm25.get_scores(query.lower().split())
    max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1
    
    combined = []
    for i, chunk in enumerate(all_chunks):
        v_score = 0
        for rank, idx in enumerate(indices[0]):
            if idx == i:
                v_score = 1 / (1 + distances[0][rank])
                break
        
        score = (alpha * v_score) + ((1 - alpha) * (bm25_scores[i] / max_bm25))
        if score > 0:
            combined.append({'chunk': chunk, 'score': score})
            
    return sorted(combined, key=lambda x: x['score'], reverse=True)[:k]

# Verification
test_query = "How many votes did Nana Addo get in the Ahafo region?"
results = hybrid_search(test_query)
for r in results:
    print(f"Score: {r['score']:.4f} | {r['chunk']['text']}\n")